# Table Format Benchmark — Iceberg vs Delta vs Hudi
**DATA 228 — Spring 2026 | Team 3**

Comparative analysis of:
- Write throughput
- Read query performance
- Storage efficiency
- Upsert/Merge operations
- Feature comparison

---

In [ ]:
import os, json
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

BENCH_DIR = os.path.join('..', 'benchmark_results')
results_path = os.path.join(BENCH_DIR, 'benchmark_results.json')

if os.path.exists(results_path):
    with open(results_path) as f:
        results = json.load(f)
    print('Benchmark results loaded')
else:
    print('Run: make benchmark')
    results = None

In [ ]:
if results:
    # Write performance
    write_data = results['write_benchmark']
    write_df = pd.DataFrame([
        {'Format': k.title(), 'Write Time (s)': v['write_time_sec'],
         'Storage (MB)': v['storage_mb']}
        for k, v in write_data.items()
    ])
    
    fig = make_subplots(rows=1, cols=2,
                        subplot_titles=['Write Time', 'Storage Size'])
    
    colors = ['#636EFA', '#EF553B', '#00CC96', '#AB63FA']
    fig.add_trace(go.Bar(x=write_df['Format'], y=write_df['Write Time (s)'],
                         marker_color=colors, showlegend=False,
                         text=write_df['Write Time (s)'],
                         textposition='outside'), row=1, col=1)
    fig.add_trace(go.Bar(x=write_df['Format'], y=write_df['Storage (MB)'],
                         marker_color=colors, showlegend=False,
                         text=write_df['Storage (MB)'],
                         textposition='outside'), row=1, col=2)
    
    fig.update_layout(height=400, title='Write Performance Comparison')
    fig.show()

In [ ]:
if results and 'read_benchmark' in results:
    read_data = results['read_benchmark']
    queries = list(next(iter(read_data.values())).keys())
    
    fig = go.Figure()
    color_map = {'parquet': '#636EFA', 'delta': '#EF553B',
                 'hudi': '#00CC96', 'iceberg': '#AB63FA'}
    
    for fmt, query_results in read_data.items():
        fig.add_trace(go.Bar(
            name=fmt.title(), x=queries,
            y=[query_results.get(q, 0) for q in queries],
            marker_color=color_map.get(fmt, '#888')
        ))
    
    fig.update_layout(
        barmode='group', title='Read Query Performance (lower is better)',
        yaxis_title='Time (seconds)', height=500,
        xaxis_tickangle=-20
    )
    fig.show()
    
    # Heatmap
    heat_df = pd.DataFrame(read_data).T
    fig2 = px.imshow(heat_df, text_auto='.3f', color_continuous_scale='RdYlGn_r',
                     title='Query Time Heatmap (seconds)', height=350)
    fig2.show()

In [ ]:
# Feature comparison matrix
features = {
    'ACID': [0, 1, 1, 1],
    'Time Travel': [0, 1, 1, 1],
    'Schema Evolution': [0.3, 1, 1, 1],
    'Upsert/Merge': [0, 1, 1, 1],
    'Partition Evolution': [0, 0, 0, 1],
    'Hidden Partitioning': [0, 0, 0, 1],
    'Streaming': [0.3, 1, 0.8, 0.8],
    'Multi-Engine': [1, 0.5, 0.7, 1],
}

feat_df = pd.DataFrame(features, index=['Parquet', 'Delta', 'Hudi', 'Iceberg'])

fig = px.imshow(feat_df, text_auto=True, color_continuous_scale='RdYlGn',
                title='Feature Support Matrix (1=Full, 0=None)', height=400)
fig.show()

In [ ]:
# Radar chart comparison
categories = ['Write Speed', 'Read Speed', 'Storage Efficiency',
              'Feature Richness', 'Ecosystem', 'Upsert Performance']

fig = go.Figure()
fig.add_trace(go.Scatterpolar(
    r=[9, 10, 10, 3, 10, 1], theta=categories, fill='toself',
    name='Parquet', line_color='#636EFA'))
fig.add_trace(go.Scatterpolar(
    r=[7, 8, 8, 8, 8, 9], theta=categories, fill='toself',
    name='Delta Lake', line_color='#EF553B'))
fig.add_trace(go.Scatterpolar(
    r=[5, 7, 7, 8, 6, 10], theta=categories, fill='toself',
    name='Apache Hudi', line_color='#00CC96'))
fig.add_trace(go.Scatterpolar(
    r=[8, 9, 9, 10, 7, 8], theta=categories, fill='toself',
    name='Apache Iceberg', line_color='#AB63FA'))

fig.update_layout(
    polar=dict(radialaxis=dict(visible=True, range=[0, 10])),
    title='Table Format Radar Comparison', height=500
)
fig.show()

## Key Takeaways

1. **Apache Iceberg** — Best for schema/partition evolution, catalog flexibility, multi-engine support
2. **Delta Lake** — Best Spark integration, native streaming, strong ecosystem
3. **Apache Hudi** — Most efficient upserts via Merge-on-Read, good for CDC workloads
4. **Parquet** — Fastest writes, smallest storage, but no ACID/time-travel/upsert support